### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
## Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

## Step 1 : Split into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

### sstep 2: Embed each setence
embeddings=model.encode(sentences)

# Step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk=[sentences[0]]

## Step 4: Semantic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")







📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


### RAG Pipeline Modular Coding

In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.schema import Document
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain.schema.runnable import RunnableLambda, RunnableMap
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [4]:
### Custom Semantic Chunker With Threshold

class ThresholdSematicChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold 

    def split(self, text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

    

In [5]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [6]:
### Chunking
chunker=ThresholdSematicChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [7]:
### VectorStore
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
embedding=OpenAIEmbeddings()
vectorstore=FAISS.from_documents(chunks,embedding)
retriever=vectorstore.as_retriever()



C:\Users\win10\AppData\Local\Temp\ipykernel_50868\2105350377.py:4: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding=OpenAIEmbeddings()


In [9]:
## Prompt Template

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [10]:
## LLM
llm=init_chat_model(model="groq:gemma2-9b-it",temperature=0.4)

### LCEL Chain With retrieval

rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

According to the provided context, LangChain is a framework for building applications with LLMs. 



### Semantic chunker With Langchain

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain.document_loaders import TextLoader

In [2]:
## Load the documents
loader=TextLoader("langchain_intro.txt")
docs=loader.load()

## Initialize embedding model
embedding=OpenAIEmbeddings()

## Create the semantic chunker
chunker=SemanticChunker(embedding)

## Split the documents
chunks=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")


 chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 chunk 2:
You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.


In [ ]:
best_practices = """
📚 SEMANTIC CHUNKING - BEST PRACTICES & USE CASES
═══════════════════════════════════════════════════════════════════════════

✅ WHEN TO USE SEMANTIC CHUNKING:

1. Multi-Topic Documents
   • Documents covering different topics
   • News articles with various subjects
   • Research papers with distinct sections

2. Conversational Text
   • Chat logs
   • Interview transcripts
   • Q&A datasets

3. Mixed-Context Content
   • Documentation with code and explanations
   • Educational content with examples
   • Technical articles with different concepts

4. RAG Applications
   • Retrieval-Augmented Generation
   • Question answering systems
   • Semantic search engines

═══════════════════════════════════════════════════════════════════════════

❌ WHEN NOT TO USE SEMANTIC CHUNKING:

1. Uniform Content
   • Single-topic articles
   • Continuous narratives
   • Simple lists

2. Strict Size Requirements
   • When chunks must be exactly N tokens
   • Fixed-size embedding requirements
   • API token limits

3. Structured Data
   • Tables and spreadsheets
   • Code with clear delimiters
   • JSON/XML documents

═══════════════════════════════════════════════════════════════════════════

🎯 THRESHOLD SELECTION GUIDE:

• High Threshold (0.8-0.9)
  → More chunks, smaller size
  → Better for diverse content
  → Higher precision, lower recall

• Medium Threshold (0.6-0.7)
  → Balanced approach
  → Good for most use cases
  → DEFAULT RECOMMENDATION

• Low Threshold (0.4-0.5)
  → Fewer, larger chunks
  → Better for related content
  → Higher recall, lower precision

═══════════════════════════════════════════════════════════════════════════

⚙️ OPTIMIZATION TIPS:

1. Start with adaptive threshold to find baseline
2. Use hierarchical chunking for multi-level retrieval
3. Add overlap for better context preservation
4. Monitor quality metrics regularly
5. A/B test with your specific use case

═══════════════════════════════════════════════════════════════════════════

🔬 ADVANCED TECHNIQUES:

• Combine with metadata filtering
• Use domain-specific embedding models
• Implement chunk post-processing
• Add custom similarity functions
• Create chunk hierarchies for navigation

═══════════════════════════════════════════════════════════════════════════
"""

print(best_practices)

## Summary: When to Use Semantic Chunking

Best practices and use cases for semantic chunking.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def visualize_semantic_chunking(text, threshold=0.7):
    """
    Visualize how semantic chunking groups sentences
    """
    # Split and embed
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(sentences)
    
    # Perform chunking to get chunk assignments
    chunker = ThresholdSematicChunker(threshold=threshold)
    chunks = chunker.split(text)
    
    # Create chunk labels for each sentence
    chunk_labels = []
    chunk_id = 0
    for chunk in chunks:
        chunk_sentences = [s.strip() for s in chunk.split('.') if s.strip() and not s.startswith('[overlap]')]
        for _ in chunk_sentences:
            chunk_labels.append(chunk_id)
        chunk_id += 1
    
    # Reduce to 2D for visualization
    if len(sentences) > 2:
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(sentences)-1))
        embeddings_2d = tsne.fit_transform(embeddings)
    else:
        embeddings_2d = embeddings[:, :2]  # Just use first 2 dimensions
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(chunks)))
    
    for i, (x, y) in enumerate(embeddings_2d):
        chunk_id = chunk_labels[i] if i < len(chunk_labels) else 0
        ax.scatter(x, y, c=[colors[chunk_id]], s=200, alpha=0.6, edgecolors='black', linewidth=1.5)
        ax.annotate(f'S{i+1}', (x, y), fontsize=10, ha='center', va='center', fontweight='bold')
    
    # Draw connections between consecutive sentences
    for i in range(len(embeddings_2d) - 1):
        x1, y1 = embeddings_2d[i]
        x2, y2 = embeddings_2d[i + 1]
        
        # Calculate similarity
        sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
        
        # Color based on similarity
        if sim >= threshold:
            color = 'green'
            linewidth = 2
            style = '-'
        else:
            color = 'red'
            linewidth = 1
            style = '--'
        
        ax.plot([x1, x2], [y1, y2], color=color, linewidth=linewidth, 
                linestyle=style, alpha=0.5, label=f'sim={sim:.2f}' if i == 0 else '')
    
    ax.set_title(f'Semantic Chunking Visualization (threshold={threshold})', 
                fontsize=16, fontweight='bold')
    ax.set_xlabel('Dimension 1', fontsize=12)
    ax.set_ylabel('Dimension 2', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', label='Same chunk (sim ≥ threshold)'),
        Patch(facecolor='red', label='Different chunks (sim < threshold)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    # Print sentence labels
    print("\n📝 Sentence Assignments:")
    print("=" * 80)
    for i, sent in enumerate(sentences):
        chunk_num = chunk_labels[i] if i < len(chunk_labels) else 0
        print(f"S{i+1} (Chunk {chunk_num + 1}): {sent[:60]}...")

# Visualize
visualize_semantic_chunking(long_text, threshold=0.7)

## New Feature 7: Visualization of Semantic Relationships

Visualize sentence embeddings and chunk boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def compare_chunking_methods(text):
    """
    Compare semantic chunking vs traditional methods
    """
    print("⚖️ Chunking Methods Comparison\n")
    print("=" * 80)
    
    # Method 1: Semantic Chunking
    semantic_chunker = ThresholdSematicChunker(threshold=0.7)
    semantic_chunks = semantic_chunker.split(text)
    
    # Method 2: Character-based
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=20,
        separators=[". ", "\n", " "]
    )
    char_chunks = char_splitter.split_text(text)
    
    # Method 3: Fixed sentence count
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    fixed_chunks = []
    for i in range(0, len(sentences), 2):
        chunk = ". ".join(sentences[i:i+2]) + "."
        fixed_chunks.append(chunk)
    
    # Compare results
    methods = [
        ("Semantic (Threshold=0.7)", semantic_chunks),
        ("Character (200 chars)", char_chunks),
        ("Fixed (2 sentences)", fixed_chunks)
    ]
    
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    for method_name, chunks in methods:
        print(f"\n📊 {method_name}")
        print("-" * 80)
        print(f"   • Number of chunks: {len(chunks)}")
        print(f"   • Avg chunk length: {np.mean([len(c) for c in chunks]):.1f} chars")
        
        # Calculate coherence
        if len(chunks) > 1:
            chunk_embs = model.encode(chunks)
            inter_sims = []
            for i in range(len(chunks) - 1):
                sim = cosine_similarity([chunk_embs[i]], [chunk_embs[i + 1]])[0][0]
                inter_sims.append(sim)
            print(f"   • Avg inter-chunk similarity: {np.mean(inter_sims):.3f}")
        
        print(f"\n   Preview of chunks:")
        for i, chunk in enumerate(chunks[:3], 1):
            print(f"   {i}. {chunk[:100]}...")
        
        if len(chunks) > 3:
            print(f"   ... and {len(chunks) - 3} more chunks")

# Run comparison
compare_chunking_methods(long_text)

## New Feature 6: Comparative Analysis - Traditional vs Semantic Chunking

Compare semantic chunking with traditional character/token-based methods.

In [ ]:
class SemanticChunkerWithOverlap:
    """
    Semantic chunker that preserves context by overlapping sentences
    """
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.7, overlap_sentences=1):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
        self.overlap_sentences = overlap_sentences
    
    def split(self, text: str):
        """Split with semantic boundaries and overlap"""
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        
        chunks = []
        i = 0
        
        while i < len(sentences):
            current_chunk = [sentences[i]]
            j = i + 1
            
            # Build chunk based on similarity
            while j < len(sentences):
                sim = cosine_similarity([embeddings[j - 1]], [embeddings[j]])[0][0]
                if sim >= self.threshold:
                    current_chunk.append(sentences[j])
                    j += 1
                else:
                    break
            
            # Add overlap from next chunk
            overlap_start = j
            overlap_end = min(j + self.overlap_sentences, len(sentences))
            for k in range(overlap_start, overlap_end):
                current_chunk.append(f"[overlap] {sentences[k]}")
            
            chunks.append(". ".join(current_chunk) + ".")
            i = j
        
        return chunks

# Test with overlap
overlap_chunker = SemanticChunkerWithOverlap(threshold=0.7, overlap_sentences=1)
overlap_chunks = overlap_chunker.split(long_text)

print("🔗 Semantic Chunking with Context Overlap\n")
print("=" * 80)
print(f"Threshold: 0.7 | Overlap: 1 sentence\n")

for i, chunk in enumerate(overlap_chunks, 1):
    print(f"\n📦 Chunk {i}:")
    print("-" * 80)
    # Highlight overlap
    chunk_display = chunk.replace("[overlap]", "🔄 [OVERLAP]")
    print(chunk_display)

## New Feature 4: Hierarchical Semantic Chunking

Create multi-level chunks for better context preservation.

In [ ]:
class AdaptiveSemanticChunker:
    """
    Automatically determines optimal threshold based on similarity distribution
    """
    def __init__(self, model_name="all-MiniLM-L6-v2", method='percentile', percentile=50):
        self.model = SentenceTransformer(model_name)
        self.method = method
        self.percentile = percentile
        self.optimal_threshold = None
    
    def _find_optimal_threshold(self, similarities):
        """Find optimal threshold using specified method"""
        if self.method == 'percentile':
            return np.percentile(similarities, self.percentile)
        elif self.method == 'mean':
            return np.mean(similarities)
        elif self.method == 'median':
            return np.median(similarities)
        elif self.method == 'kmeans':
            # Use gap between high and low similarities
            sorted_sims = sorted(similarities)
            gaps = [sorted_sims[i+1] - sorted_sims[i] for i in range(len(sorted_sims)-1)]
            max_gap_idx = gaps.index(max(gaps))
            return (sorted_sims[max_gap_idx] + sorted_sims[max_gap_idx + 1]) / 2
        else:
            return 0.7  # default
    
    def split(self, text: str):
        """Split text with adaptive thresholding"""
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        
        if len(sentences) < 2:
            return [text]
        
        embeddings = self.model.encode(sentences)
        
        # Calculate all pairwise similarities
        all_similarities = []
        for i in range(len(sentences) - 1):
            sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
            all_similarities.append(sim)
        
        # Find optimal threshold
        self.optimal_threshold = self._find_optimal_threshold(all_similarities)
        
        # Perform chunking with optimal threshold
        chunks = []
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.optimal_threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]
        
        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def get_threshold_info(self):
        """Get information about the selected threshold"""
        return {
            'method': self.method,
            'optimal_threshold': self.optimal_threshold
        }

# Test different adaptive methods
methods = ['percentile', 'mean', 'median', 'kmeans']

print("🎯 Adaptive Semantic Chunking - Method Comparison\n")
print("=" * 80)

for method in methods:
    print(f"\n📊 Method: {method.upper()}")
    print("-" * 80)
    
    chunker = AdaptiveSemanticChunker(method=method, percentile=60)
    chunks = chunker.split(test_text)
    info = chunker.get_threshold_info()
    
    print(f"   • Optimal Threshold: {info['optimal_threshold']:.3f}")
    print(f"   • Number of Chunks: {len(chunks)}")
    print(f"   • Avg Chunk Length: {np.mean([len(c) for c in chunks]):.0f} chars")
    
    print(f"\n   Chunks:")
    for i, chunk in enumerate(chunks, 1):
        print(f"   {i}. {chunk[:80]}...")